# CPDS-AI: YOLOv8 Adult vs Child Classification

Notebook này được thiết kế để chạy trên Kaggle/Google Colab. Mục tiêu là tải dữ liệu từ Roboflow, fine-tune mô hình **YOLOv8n** trên tập dữ liệu Adult/Child, sau đó xuất ra file ONNX.

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 1. Tải Dataset từ Roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="[REDACTED]")
project = rf.workspace("timii-owolabi-pwfjm").project("child-adult-detection-bgjzk")
version = project.version(10)
dataset = version.download("yolov8")

print("Đường dẫn dataset:", dataset.location)

## 2. Huấn luyện (Train) YOLOv8n

In [ ]:
from ultralytics import YOLO

# Load pre-trained model nano
model = YOLO('yolov8n.pt')

# Kaggle GPU: device=0
results = model.train(data=dataset.location + '/data.yaml', epochs=20, imgsz=640, device=0, project="/kaggle/working/runs", name="adult_child", exist_ok=True)

## 3. Chuyển đổi định dạng sang ONNX (Export)

File `best.onnx` sẽ được lưu ở cột tay phải của Kaggle, trong thư mục `/kaggle/working/runs/adult_child/weights/best.onnx`

In [ ]:
# Load best weights
best_model = YOLO('/kaggle/working/runs/adult_child/weights/best.pt')

# Export sang ONNX format
export_path = best_model.export(format='onnx', opset=12, dynamic=True, simplify=True)
print(f"Model exported to: {export_path}")